# 🏔️ Himalayan Landslide Detection — Improved Training

### What's fixed vs original:
| Issue | Original | This notebook |
|---|---|---|
| Loss function | CrossEntropy only | **Dice + Focal** (handles class imbalance) |
| Normalisation | None | **Per-band z-score** (matches Landslide4Sense paper) |
| Augmentation | None | **Flip + rotate + brightness jitter** |
| Epochs | 15 | **40 with early stopping** |
| LR schedule | Fixed 1e-4 | **CosineAnnealing + AdamW** |
| Class imbalance | Ignored | **WeightedSampler + Focal loss** |
| Checkpoint metric | Dice | **IoU** (more meaningful for sparse masks) |

In [ ]:
# Step 1: Kaggle setup
from google.colab import files
files.upload()  # upload kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle h5py tqdm

In [ ]:
# Step 2: Download and extract dataset
!kaggle datasets download -d tekbahadurkshetri/landslide4sense
!unzip -q landslide4sense.zip -d /content/landslide4sense/
import os
print(os.listdir('/content/landslide4sense/'))

In [ ]:
# Step 3: Explore dataset and check class balance
import h5py, numpy as np, os
import matplotlib.pyplot as plt

train_img_dir  = '/content/landslide4sense/TrainData/img/'
train_mask_dir = '/content/landslide4sense/TrainData/mask/'
img_files = sorted([f for f in os.listdir(train_img_dir) if f.endswith('.h5')])
print(f'Training samples: {len(img_files)}')

# Check class balance on first 200 samples
ls_px, total_px = 0, 0
for fname in img_files[:200]:
    mname = fname.replace('image_', 'mask_')
    with h5py.File(os.path.join(train_mask_dir, mname), 'r') as f:
        mask = np.array(f['mask'])
    ls_px    += (mask == 1).sum()
    total_px += mask.size

ls_pct = ls_px / total_px * 100
print(f'Landslide pixels : {ls_pct:.2f}%')
print(f'Background pixels: {100-ls_pct:.2f}%')
print('Heavy class imbalance — this is why plain CrossEntropy failed!')

# Visualise a sample
with h5py.File(os.path.join(train_img_dir, img_files[5]), 'r') as f:
    img = np.array(f['img'])
with h5py.File(os.path.join(train_mask_dir, img_files[5].replace('image_','mask_')), 'r') as f:
    mask = np.array(f['mask'])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
rgb = img[:, :, [2,1,0]]
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
axes[0].imshow(rgb);                          axes[0].set_title('RGB (B3,B2,B1)');   axes[0].axis('off')
axes[1].imshow(img[:,:,3],  cmap='RdYlGn');   axes[1].set_title('NIR (band 4)');     axes[1].axis('off')
axes[2].imshow(img[:,:,11], cmap='gray');      axes[2].set_title('SAR-VV (band 12)'); axes[2].axis('off')
axes[3].imshow(mask,        cmap='Reds');      axes[3].set_title('Mask (1=landslide)');axes[3].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Compute per-band statistics for z-score normalisation
# THE most important fix — without this, optical bands (0-1 range) and
# SAR/Slope bands (different scales) confuse the model completely.
import h5py, numpy as np, os
from tqdm import tqdm

train_img_dir = '/content/landslide4sense/TrainData/img/'
img_files = sorted([f for f in os.listdir(train_img_dir) if f.endswith('.h5')])
N_BANDS = 14
BAND_NAMES = ['B1(Blue)','B2(Green)','B3(Red)','B4(NIR)','B5(SWIR1)','B6(SWIR2)',
              'B7(RE1)','B8(RE2)','B9(RE3)','B10(NIR08)','B11(SWIR22)',
              'B12(SAR-VV)','B13(SAR-VH)','B14(Slope)']

# Two-pass mean/std
mean = np.zeros(N_BANDS, dtype=np.float64)
for fname in tqdm(img_files, desc='Pass 1/2: mean'):
    with h5py.File(os.path.join(train_img_dir, fname), 'r') as f:
        img = np.array(f['img']).astype(np.float64)
    mean += img.reshape(-1, N_BANDS).mean(axis=0)
mean /= len(img_files)

var = np.zeros(N_BANDS, dtype=np.float64)
for fname in tqdm(img_files, desc='Pass 2/2: std'):
    with h5py.File(os.path.join(train_img_dir, fname), 'r') as f:
        img = np.array(f['img']).astype(np.float64)
    var += ((img.reshape(-1, N_BANDS) - mean) ** 2).mean(axis=0)
var /= len(img_files)
std  = np.sqrt(var)

print('\n=== Per-band statistics ===')
for i, (m, s, name) in enumerate(zip(mean, std, BAND_NAMES)):
    print(f'  Band {i+1:2d} {name:14s}: mean={m:.4f}  std={s:.4f}')

BAND_MEAN = mean.astype(np.float32)
BAND_STD  = std.astype(np.float32)
np.save('/content/band_mean.npy', BAND_MEAN)
np.save('/content/band_std.npy',  BAND_STD)
print('\nSaved band_mean.npy + band_std.npy')

In [ ]:
# Step 5: Dataset with proper normalisation and augmentation
import torch, h5py, numpy as np, os, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

BAND_MEAN = np.load('/content/band_mean.npy')
BAND_STD  = np.load('/content/band_std.npy')

class LandslideDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.augment   = augment
        self.img_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.h5')])
        print(f'Loaded {len(self.img_files)} samples | augment={augment}')

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        fname = self.img_files[idx]
        mname = fname.replace('image_', 'mask_')
        with h5py.File(os.path.join(self.img_dir,  fname), 'r') as f:
            img  = np.array(f['img']).astype(np.float32)   # (128,128,14)
        with h5py.File(os.path.join(self.mask_dir, mname), 'r') as f:
            mask = np.array(f['mask']).astype(np.int64)    # (128,128)
        # Per-band z-score normalisation
        img = (img - BAND_MEAN) / (BAND_STD + 1e-8)
        # Augmentation (training only)
        if self.augment:
            if random.random() > 0.5:
                img  = np.fliplr(img).copy();  mask = np.fliplr(mask).copy()
            if random.random() > 0.5:
                img  = np.flipud(img).copy();  mask = np.flipud(mask).copy()
            k = random.randint(0, 3)
            if k > 0:
                img  = np.rot90(img,  k=k).copy()
                mask = np.rot90(mask, k=k).copy()
            if random.random() > 0.5:
                img[:, :, :11] *= random.uniform(0.85, 1.15)
        img  = torch.from_numpy(img.transpose(2, 0, 1))  # (14,128,128)
        mask = torch.from_numpy(mask)
        return img, mask

    def get_sample_weights(self):
        weights = []
        for fname in self.img_files:
            mname = fname.replace('image_', 'mask_')
            with h5py.File(os.path.join(self.mask_dir, mname), 'r') as f:
                mask = np.array(f['mask'])
            # Landslide-rich images sampled more often
            weights.append(max(float((mask==1).mean()) * 10 + 0.1, 0.1))
        return weights

print('Dataset class ready.')

In [ ]:
# Step 6: Loss functions  (Focal + Dice combined)
import torch, torch.nn as nn, torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs  = torch.softmax(logits, dim=1)[:, 1]
        target = (targets == 1).float()
        inter  = (probs * target).sum(dim=(1,2))
        union  = probs.sum(dim=(1,2)) + target.sum(dim=(1,2))
        return 1.0 - ((2*inter + self.smooth) / (union + self.smooth)).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        at = torch.where(targets==1,
                         torch.full_like(ce, self.alpha),
                         torch.full_like(ce, 1-self.alpha))
        return (at * (1-pt)**self.gamma * ce).mean()

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.focal = FocalLoss(alpha=0.75, gamma=2.0)
        self.dice  = DiceLoss(smooth=1.0)
    def forward(self, logits, targets):
        return 0.5 * self.focal(logits, targets) + 0.5 * self.dice(logits, targets)

print('Loss functions ready.')

In [ ]:
# Step 7: Metrics (Dice, IoU, Precision, Recall, F1)
import torch

def compute_metrics(logits, targets, threshold=0.5):
    probs  = torch.softmax(logits, dim=1)[:, 1]
    preds  = (probs > threshold).float()
    target = (targets == 1).float()
    tp = (preds * target).sum().item()
    fp = (preds * (1-target)).sum().item()
    fn = ((1-preds) * target).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2*precision*recall / (precision+recall+1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)
    dice      = 2*tp / (2*tp + fp + fn + 1e-8)
    return {'dice': dice, 'iou': iou, 'precision': precision, 'recall': recall, 'f1': f1}

print('Metrics ready.')

In [ ]:
# Step 8: U-Net with bottleneck dropout
import torch, torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if dropout > 0: layers.append(nn.Dropout2d(p=dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, n_channels=14, n_classes=2):
        super().__init__()
        self.enc1 = DoubleConv(n_channels, 64)
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64,  128))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.bot  = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024, dropout=0.3))
        self.up1  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec1 = DoubleConv(1024, 512)
        self.up2  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec2 = DoubleConv(512, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        self.up4  = nn.ConvTranspose2d(128,  64, 2, stride=2)
        self.dec4 = DoubleConv(128,  64)
        self.out  = nn.Conv2d(64, n_classes, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bot(e4)
        d  = self.dec1(torch.cat([self.up1(b),  e4], dim=1))
        d  = self.dec2(torch.cat([self.up2(d),  e3], dim=1))
        d  = self.dec3(torch.cat([self.up3(d),  e2], dim=1))
        d  = self.dec4(torch.cat([self.up4(d),  e1], dim=1))
        return self.out(d)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = UNet(14, 2).to(device)
dummy  = torch.randn(2, 14, 128, 128).to(device)
assert model(dummy).shape == (2, 2, 128, 128), 'Shape mismatch!'
print(f'U-Net ready on {device} | Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Step 9: Full training loop
import torch, torch.optim as optim, numpy as np, os, time
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm

TRAIN_IMG  = '/content/landslide4sense/TrainData/img/'
TRAIN_MASK = '/content/landslide4sense/TrainData/mask/'
VALID_IMG  = '/content/landslide4sense/ValidData/img/'
VALID_MASK = '/content/landslide4sense/ValidData/mask/'
CKPT_DIR   = '/content/Himalayan_Landslide_Detection/outputs/checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)

EPOCHS, BATCH_SIZE, LR, PATIENCE = 40, 16, 1e-3, 8

train_ds = LandslideDataset(TRAIN_IMG, TRAIN_MASK, augment=True)
val_ds   = LandslideDataset(VALID_IMG, VALID_MASK, augment=False)

sampler      = WeightedRandomSampler(train_ds.get_sample_weights(), len(train_ds), replacement=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)
print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')

model     = UNet(14, 2).to(device)
criterion = CombinedLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {'train_loss':[], 'val_loss':[], 'val_dice':[], 'val_iou':[]}
best_iou, patience_ctr = 0.0, 0
best_path = os.path.join(CKPT_DIR, 'best_unet.pth')

print(f'Training on {device} | max {EPOCHS} epochs\n')
t0 = time.time()

for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Ep {epoch:02d}/{EPOCHS}', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, agg = 0.0, {'dice':0,'iou':0,'precision':0,'recall':0,'f1':0}
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, masks).item()
            m = compute_metrics(logits, masks)
            for k in agg: agg[k] += m[k]
    n = len(val_loader)
    val_loss /= n
    for k in agg: agg[k] /= n
    scheduler.step()

    tl = train_loss / len(train_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(agg['dice'])
    history['val_iou'].append(agg['iou'])

    lr_now = optimizer.param_groups[0]['lr']
    print(f"Ep {epoch:02d} | Loss {tl:.4f}/{val_loss:.4f} | "
          f"Dice {agg['dice']:.4f} | IoU {agg['iou']:.4f} | "
          f"P {agg['precision']:.3f} R {agg['recall']:.3f} | LR {lr_now:.2e}")

    if agg['iou'] > best_iou:
        best_iou = agg['iou']
        torch.save(model.state_dict(), best_path)
        print(f'  ✅ Best IoU={best_iou:.4f} saved')
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nDone in {(time.time()-t0)/60:.1f} min | Best IoU: {best_iou:.4f}')

In [ ]:
# Step 10: Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], label='Train loss', color='#e74c3c')
axes[0].plot(ep, history['val_loss'],   label='Val loss',   color='#3498db')
axes[0].set_title('Combined Focal + Dice Loss')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['val_dice'], label='Val Dice', color='#2ecc71')
axes[1].plot(ep, history['val_iou'],  label='Val IoU',  color='#9b59b6')
axes[1].axhline(y=best_iou, color='#9b59b6', linestyle='--',
                alpha=0.5, label=f'Best IoU={best_iou:.4f}')
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/Himalayan_Landslide_Detection/outputs/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 11: Visual predictions on validation set
import matplotlib.pyplot as plt, numpy as np, torch

model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

fig, axes = plt.subplots(6, 4, figsize=(16, 24))
for ax, t in zip(axes[0], ['RGB', 'NIR band', 'Ground truth', 'Prediction (prob)']):
    ax.set_title(t, fontsize=11, fontweight='bold')

shown = 0
with torch.no_grad():
    for imgs, masks in val_loader:
        logits = model(imgs.to(device))
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        for b in range(imgs.shape[0]):
            if shown >= 6: break
            if masks[b].numpy().sum() < 50: continue  # skip near-empty
            img_np = imgs[b].numpy()  # (14,128,128) normalised
            rgb = img_np[[2,1,0]] * BAND_STD[[2,1,0],None,None] + BAND_MEAN[[2,1,0],None,None]
            rgb = np.clip(rgb.transpose(1,2,0), 0, 1)
            nir = img_np[3] * BAND_STD[3] + BAND_MEAN[3]
            m = compute_metrics(logits[b:b+1], masks[b:b+1].to(device))
            axes[shown,0].imshow(rgb);                          axes[shown,0].axis('off')
            axes[shown,1].imshow(nir, cmap='RdYlGn');          axes[shown,1].axis('off')
            axes[shown,2].imshow(masks[b].numpy(), cmap='Reds');axes[shown,2].axis('off')
            axes[shown,3].imshow(probs[b], cmap='hot', vmin=0, vmax=1)
            axes[shown,3].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}", fontsize=9)
            axes[shown,3].axis('off')
            shown += 1
        if shown >= 6: break

plt.suptitle('Best model predictions on validation set', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/content/Himalayan_Landslide_Detection/outputs/val_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 12: Save band stats alongside the model weights
# app.py will load these at startup so inference normalisation
# exactly matches how the model was trained.
import numpy as np, os

ckpt_dir = '/content/Himalayan_Landslide_Detection/outputs/checkpoints/'
np.save(os.path.join(ckpt_dir, 'band_mean.npy'), BAND_MEAN)
np.save(os.path.join(ckpt_dir, 'band_std.npy'),  BAND_STD)
print('Saved band_mean.npy + band_std.npy')
print(f'Best IoU : {best_iou:.4f}')
print(f'Best Dice: {max(history["val_dice"]):.4f}')

In [ ]:
# Step 13: Download outputs
from google.colab import files

!zip -r /content/landslide_model_output.zip \
    /content/Himalayan_Landslide_Detection/outputs/checkpoints/ \
    /content/Himalayan_Landslide_Detection/outputs/training_curves.png \
    /content/Himalayan_Landslide_Detection/outputs/val_predictions.png

files.download('/content/landslide_model_output.zip')
print('Extract the zip and place:')
print('  best_unet.pth   -> outputs/checkpoints/')
print('  band_mean.npy   -> outputs/checkpoints/')
print('  band_std.npy    -> outputs/checkpoints/')